# Tree of Thought: Search Over Reasoning Paths

Tree of Thought (ToT, Yao et al. 2023) generalizes chain-of-thought by treating reasoning as a search problem. Instead of generating a single chain, ToT maintains a tree of partial solutions and uses the model to evaluate and select the most promising branches.

## From Chain to Tree

CoT generates a linear sequence: thought_1 → thought_2 → ... → answer. This commits to each step greedily and cannot backtrack when an early step turns out to be wrong.

ToT introduces:
- **Thought decomposition**: break the problem into steps where each step is a 'thought' — a coherent partial solution
- **Thought generation**: generate K candidate thoughts at each node (branching factor)
- **State evaluation**: use the model to score each partial solution (promising / sure / impossible)
- **Search algorithm**: BFS or DFS over the tree, guided by the evaluation scores

This enables systematic exploration and backtracking — capabilities that pure CoT lacks.

## When ToT Helps vs When It Doesn't

ToT provides the most benefit for problems where:
- Early decisions significantly constrain later options (combinatorial problems)
- There are a small number of valid solution strategies that need to be explored
- The model can reliably evaluate partial solutions (not just final answers)

ToT provides little benefit for:
- Straightforward reasoning chains (arithmetic, factual lookup) — the overhead outweighs the benefit
- Problems where evaluation is as hard as generation (can't verify partial solutions)
- Real-time applications — ToT is expensive, making it unsuitable for latency-sensitive contexts

The original paper showed ToT on creative writing (coherent 4-paragraph stories) and crosswords, where structured exploration was essential.

In [ ]:
from dataclasses import dataclass, field
from typing import Callable, Optional
import random

@dataclass
class ThoughtNode:
    state: str
    depth: int
    score: float = 0.0
    parent: Optional['ThoughtNode'] = None
    children: list = field(default_factory=list)
    terminal: bool = False

    def path_to_root(self) -> list:
        path = []
        node = self
        while node:
            path.append(node.state)
            node = node.parent
        return list(reversed(path))

class TreeOfThought:
    def __init__(self, generate_fn: Callable, evaluate_fn: Callable,
                 branch_factor: int = 3, max_depth: int = 4,
                 strategy: str = 'bfs'):
        self.generate = generate_fn
        self.evaluate = evaluate_fn
        self.branch_factor = branch_factor
        self.max_depth = max_depth
        self.strategy = strategy
        self.nodes_explored = 0

    def solve(self, problem: str) -> tuple:
        root = ThoughtNode(state=problem, depth=0)
        if self.strategy == 'bfs':
            return self._bfs(root)
        return self._dfs(root)

    def _bfs(self, root: ThoughtNode) -> tuple:
        frontier = [root]
        best = None
        for _ in range(self.max_depth):
            next_frontier = []
            for node in frontier:
                thoughts = self.generate(node.state, self.branch_factor)
                for thought in thoughts:
                    child = ThoughtNode(state=thought, depth=node.depth+1, parent=node)
                    child.score = self.evaluate(thought, node.depth+1)
                    node.children.append(child)
                    self.nodes_explored += 1
                    if best is None or child.score > best.score:
                        best = child
                    if child.depth < self.max_depth:
                        next_frontier.append(child)
            # Keep top-K by score for next level
            frontier = sorted(next_frontier, key=lambda n: -n.score)[:self.branch_factor]
        return best, best.path_to_root() if best else []

    def _dfs(self, node: ThoughtNode, depth: int = 0) -> tuple:
        if depth >= self.max_depth:
            return node, node.path_to_root()
        thoughts = self.generate(node.state, self.branch_factor)
        best = node
        best_path = node.path_to_root()
        for thought in thoughts:
            child = ThoughtNode(state=thought, depth=depth+1, parent=node)
            child.score = self.evaluate(thought, depth+1)
            node.children.append(child)
            self.nodes_explored += 1
            if child.score > 0.5:
                result, path = self._dfs(child, depth+1)
                if result.score > best.score:
                    best, best_path = result, path
        return best, best_path

# Mock functions for a planning problem
rng = random.Random(42)

def generate_thoughts(state: str, k: int) -> list:
    templates = [
        f'First, identify the key constraint: {state[:30]}...',
        f'Approach via decomposition: break into sub-problems...',
        f'Check boundary conditions: what happens at extremes...',
        f'Consider the inverse problem: work backwards from goal...',
        f'Apply pattern matching: this resembles a known problem type...',
    ]
    return rng.sample(templates, min(k, len(templates)))

def evaluate_thought(thought: str, depth: int) -> float:
    # Heuristic: longer, more specific thoughts score higher
    base = min(0.9, len(thought) / 200)
    depth_bonus = 0.05 * depth
    return min(1.0, base + depth_bonus + rng.gauss(0, 0.05))

tot = TreeOfThought(generate_thoughts, evaluate_thought, branch_factor=3, max_depth=3, strategy='bfs')
best_node, path = tot.solve('How many ways can 5 people sit at a round table?')
print(f'Nodes explored: {tot.nodes_explored}')
print(f'Best node score: {best_node.score:.3f}')
print('Reasoning path:')
for i, step in enumerate(path):
    print(f'  Step {i}: {step[:80]}')

## BFS vs DFS for Reasoning

**BFS**: explores all branches at each depth level before going deeper. Guarantees finding the shallowest solution but requires keeping all frontier nodes in memory. Better when you expect solutions at a specific depth.

**DFS with pruning**: commits to one branch first, backtracks when stuck. Lower memory requirement. Better for problems where deep exploration of promising branches is more valuable than breadth.

**Beam search** (a hybrid): keep the top-K nodes at each depth level (BFS with pruning). The most practical approach — controls the tradeoff between breadth and memory/cost.

## Practical Limitations

ToT's appeal in the research setting does not always transfer to production:

**Cost**: with branch factor K=5 and depth D=4, ToT makes up to 5^4 = 625 model calls per problem. Even with aggressive pruning, this is 10-50x more expensive than CoT.

**Evaluation quality**: the model must reliably evaluate partial solutions. For many real tasks (open-ended writing, complex coding), intermediate evaluation is as hard as final evaluation.

**Benchmark inflation**: ToT was shown on tasks that were specifically designed to benefit from search. Performance improvements are much smaller on benchmarks like GSM8K where CoT with self-consistency already performs well.

For most production use cases, self-consistency over CoT achieves 80% of ToT's benefit at 20% of the cost.